In [1]:
!pip install transformers datasets accelerate tensorboard evaluate --upgrade

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import evaluate
from datasets import load_dataset
from transformers import (
    AutoImageProcessor,
    AutoModelForImageClassification,
    MobileNetV2Config,
    MobileNetV2ForImageClassification,
    Trainer,
    TrainingArguments,
    DefaultDataCollator
)

# Load the dataset
dataset = load_dataset("AI-Lab-Makerere/beans")

# Load processor and process the dataset
teacher_processor = AutoImageProcessor.from_pretrained("merve/beans-vit-224")

def process(examples):
    return teacher_processor(examples["image"])

processed_datasets = dataset.map(process, batched=True)
num_labels = len(processed_datasets["train"].features["labels"].names)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 74.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 24.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 24.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 17.6 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
  Attempting uninstall: accelerate
    Found existing installation: accelerate 1.13.0
    Uninstalling accelerate-1.13.0:
      Successfully uninstalled accelerate-1.13.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.10.2
    Uninstalling trans

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/4.95k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/144M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/18.5M [00:00<?, ?B/s]

...

Generating test split:   0%|          | 0/128 [00:00<?, ? examples/s]

preprocessor_config.json:   0%|          | 0.00/325 [00:00<?, ?B/s]

Map:   0%|          | 0/1034 [00:00<?, ? examples/s]

Map:   0%|          | 0/133 [00:00<?, ? examples/s]

Map:   0%|          | 0/128 [00:00<?, ? examples/s]

In [2]:
class ImageDistilTrainer(Trainer):
    def __init__(self, teacher_model=None, student_model=None, temperature=None, lambda_param=None, *args, **kwargs):
        super().__init__(model=student_model, *args, **kwargs)
        self.teacher = teacher_model
        self.student = student_model
        self.loss_function = nn.KLDivLoss(reduction="batchmean")

        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.teacher.to(device)
        self.teacher.eval()

        self.temperature = temperature
        self.lambda_param = lambda_param

    def compute_loss(self, student, inputs, return_outputs=False, **kwargs):
        student_output = self.student(**inputs)

        with torch.no_grad():
            teacher_output = self.teacher(**inputs)

        soft_teacher = F.softmax(teacher_output.logits / self.temperature, dim=-1)
        soft_student = F.log_softmax(student_output.logits / self.temperature, dim=-1)

        distillation_loss = self.loss_function(soft_student, soft_teacher) * (self.temperature ** 2)
        student_target_loss = student_output.loss

        loss = (1. - self.lambda_param) * student_target_loss + self.lambda_param * distillation_loss

        return (loss, student_output) if return_outputs else loss

In [3]:
training_args = TrainingArguments(
    output_dir="distilled-mobilenet-beans",
    num_train_epochs=30,
    fp16=True,
    logging_strategy="epoch",
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    report_to="tensorboard"
)

# Load Teacher
teacher_model = AutoModelForImageClassification.from_pretrained(
    "merve/beans-vit-224",
    num_labels=num_labels,
    ignore_mismatched_sizes=True
)

# Load Student (MobileNetV2)
student_config = MobileNetV2Config()
student_config.num_labels = num_labels
student_model = MobileNetV2ForImageClassification(student_config)

# Metrics
accuracy = evaluate.load("accuracy")
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    acc = accuracy.compute(references=labels, predictions=np.argmax(predictions, axis=1))
    return {"accuracy": acc["accuracy"]}

config.json:   0%|          | 0.00/799 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/343M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/343M [00:00<?, ?B/s]

In [4]:
trainer = ImageDistilTrainer(
    student_model=student_model,
    teacher_model=teacher_model,
    args=training_args,
    train_dataset=processed_datasets["train"],
    eval_dataset=processed_datasets["validation"],
    data_collator=DefaultDataCollator(),
    processing_class=teacher_processor,
    compute_metrics=compute_metrics,
    temperature=5,
    lambda_param=0.5
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.873816,2.262809,0.338346
2,0.826386,1.054590,0.473684
3,0.692477,1.433650,0.496241
4,0.658467,0.984703,0.458647
5,0.633810,1.737959,0.511278
6,0.604786,1.032493,0.661654
7,0.578924,1.156692,0.586466
8,0.573527,1.309706,0.601504
9,0.551765,2.302538,0.413534
10,0.553346,0.978552,0.586466


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3900, training_loss=0.49523414220565404, metrics={'train_runtime': 3645.6327, 'train_samples_per_second': 8.509, 'train_steps_per_second': 1.07, 'total_flos': 6.24122679527424e+16, 'train_loss': 0.49523414220565404, 'epoch': 30.0})

In [5]:
# Evaluate Teacher
teacher_eval_trainer = Trainer(model=teacher_model, eval_dataset=processed_datasets["test"], compute_metrics=compute_metrics)
teacher_accuracy = teacher_eval_trainer.evaluate()['eval_accuracy'] * 100

# Evaluate Untrained Student
untrained_student = MobileNetV2ForImageClassification(student_config)
student_before_accuracy = Trainer(model=untrained_student, eval_dataset=processed_datasets["test"], compute_metrics=compute_metrics).evaluate()['eval_accuracy'] * 100

# Evaluate Distilled Student
student_after_accuracy = trainer.evaluate(processed_datasets["test"])['eval_accuracy'] * 100

print(f"{'Model Setup':<30} | {'Test Accuracy':<15}")
print("-" * 50)
print(f"{'Teacher (ViT-base)':<30} | {teacher_accuracy:.2f}%")
print(f"{'Student Before KD':<30} | {student_before_accuracy:.2f}%")
print(f"{'Student After KD':<30} | {student_after_accuracy:.2f}%")

Training Loss,Validation Loss,Step,Accuracy
No log,0.334754,0,0.937500


Training Loss,Validation Loss,Step,Accuracy
No log,1.098612,0,0.335938


Training Loss,Validation Loss,Epoch,Accuracy
0.362486,0.878023,30,0.593750


Model Setup                    | Test Accuracy  
--------------------------------------------------
Teacher (ViT-base)             | 93.75%
Student Before KD              | 33.59%
Student After KD               | 59.38%
